In [ ]:
_NB_NAME_ = "Statistical Learning"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


# 📐 Statistical Learning
**ISLP Chapter 2 · Pattern Recognition for the Rest of Us**

> The framework that underlies everything else in this course. Before building models, we need to understand *what* we're trying to do and *why* it works.

---
### What you'll learn
- The difference between supervised and unsupervised learning
- Prediction vs inference — two very different goals
- Why all models have irreducible error (and what that means practically)
- How to measure model accuracy with MSE

### Prerequisites
- Basic Python (lists, loops, functions)
- Some familiarity with scatter plots

### Time
~45 minutes

## 🎯 Part 1 — What is Statistical Learning?

Statistical learning is about finding a function *f* that relates inputs **X** (predictors, features) to an output **Y** (response, target):

```
Y = f(X) + ε
```

- **f(X)** = the systematic information X provides about Y (what we can learn)
- **ε** = irreducible error — random noise that no model can capture

We never observe f directly. We *estimate* it from data. That estimate is written **f̂**.

**Two reasons to estimate f:**
1. **Prediction** — we want accurate Ŷ values. We don't care *why*, just *how well*.
2. **Inference** — we want to understand *how* Y changes with X. Which predictors matter? What's the relationship?

These goals lead to different model choices. A black-box neural network may predict well but explain nothing. A linear model may explain everything but predict poorly on complex data.

In [ ]:
# ── Setup: install and import everything we need ──────────────────────────────
# All packages below are pre-installed in Google Colab — no pip install needed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print("✓ All packages loaded")

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Advertising

In [ ]:
import pandas as pd
ads = pd.read_csv(f'{DATA_DIR}/Advertising.csv')
print(f"Advertising: {ads.shape}  |  Columns: {list(ads.columns)}")
ads.head()

## 📊 Part 2 — A Concrete Example: Advertising & Sales

We'll use the classic **Advertising dataset** from ISLP.

- **Predictors (X):** TV, radio, and newspaper advertising budgets (in $thousands)
- **Response (Y):** Sales (in thousands of units)
- **Goal:** Can we predict sales from advertising spend? Which channels matter most?

This dataset has 200 observations — one row per market.

In [ ]:
# Summary statistics — always start here
print("=== Summary Statistics ===\n")
print(ads.describe().round(2))

In [ ]:
# Visualize: Sales vs each advertising channel
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
channels = ['TV', 'radio', 'newspaper']
colors = ['#1e5fa8', '#e85d20', '#1a7a45']

for ax, col, color in zip(axes, channels, colors):
    ax.scatter(ads[col], ads['sales'], alpha=0.5, color=color, s=30, edgecolors='white', lw=0.5)
    # Fit a simple line to show the trend
    m, b = np.polyfit(ads[col], ads['sales'], 1)
    x_line = np.linspace(ads[col].min(), ads[col].max(), 100)
    ax.plot(x_line, m*x_line + b, color='black', lw=1.5, alpha=0.7)
    ax.set_xlabel(f'{col} budget ($000s)')
    ax.set_ylabel('Sales (000 units)' if col=='TV' else '')
    ax.set_title(f'{col} vs Sales\nr = {ads[col].corr(ads["sales"]):.2f}')

plt.suptitle('Advertising Dataset — Sales vs Each Channel', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\n📌 Observation: TV has the strongest linear relationship with sales (r=0.78).")
print("   Newspaper barely correlates. Radio is moderate.")

## ⚙️ Part 3 — Reducible vs Irreducible Error

The expected prediction error for any estimate f̂ can be split into two parts:

```
E[(Y - Ŷ)²] = [Reducible error] + [Irreducible error]
             = [f(X) - f̂(X)]²  +  Var(ε)
```

**Reducible error** — comes from f̂ being a poor approximation of f. *We can reduce this by choosing better models or using more data.*

**Irreducible error** — comes from ε, which is random noise in the data-generating process (unmeasured variables, randomness, measurement error). *No model, no matter how complex, can eliminate this.*

This is a fundamental limit on predictive accuracy. The best we can do is drive reducible error toward zero — we can never touch ε.

In [ ]:
# Simulate: demonstrate reducible vs irreducible error
np.random.seed(42)
n = 150

# True underlying function: f(X) = 3 + 2X - 0.5X²
X = np.linspace(0, 4, n)
f_true = 3 + 2*X - 0.5*X**2          # true f(X) — never observed
epsilon = np.random.normal(0, 0.8, n)  # irreducible error ε
Y = f_true + epsilon                   # observed Y = f(X) + ε

# Fit three models with increasing flexibility
models = {
    'Linear (underfit)': Pipeline([('poly', PolynomialFeatures(1)), ('lr', LinearRegression())]),
    'Quadratic (good fit)': Pipeline([('poly', PolynomialFeatures(2)), ('lr', LinearRegression())]),
    'Degree-15 (overfit)': Pipeline([('poly', PolynomialFeatures(15)), ('lr', LinearRegression())]),
}

X_2d = X.reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
colors_m = ['#e85d20', '#1a7a45', '#c0392b']

for ax, (name, model), color in zip(axes, models.items(), colors_m):
    model.fit(X_2d, Y)
    Y_pred = model.predict(X_2d)
    
    train_mse = mean_squared_error(Y, Y_pred)
    irred = np.var(epsilon)
    reducible = max(0, train_mse - irred)
    
    ax.scatter(X, Y, alpha=0.3, s=20, color='#888', label='Observed data')
    ax.plot(X, f_true, 'k--', lw=1.5, label='True f(X)', alpha=0.7)
    ax.plot(X, Y_pred, color=color, lw=2.5, label=f'f̂(X)')
    ax.set_title(f'{name}\nMSE={train_mse:.2f}  |  Irred≈{irred:.2f}')
    ax.set_xlabel('X')
    if ax == axes[0]: ax.set_ylabel('Y')
    ax.legend(fontsize=8)

plt.suptitle('Reducible vs Irreducible Error — Three Model Flexibilities', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

print("\n📌 The quadratic model (middle) gets closest to the true f(X).")
print("   The degree-15 model memorizes noise — low training error but will fail on new data.")
print(f"   Irreducible error ≈ {np.var(epsilon):.2f} — no model can go below this floor.")

## 📏 Part 4 — Measuring Accuracy: MSE

For regression (continuous Y), the most common accuracy metric is **Mean Squared Error (MSE)**:

```
MSE = (1/n) × Σ(yᵢ - f̂(xᵢ))²
```

But there's a critical distinction:
- **Training MSE** — error on the data used to fit the model. Always goes down as complexity increases.
- **Test MSE** — error on *new, unseen data*. This is what we actually care about.

A model with very low training MSE but high test MSE is **overfitting** — it has memorized the training data instead of learning the underlying pattern.

In [ ]:
# Show training vs test MSE as model complexity increases
np.random.seed(0)
n_train, n_test = 80, 40
X_all = np.sort(np.random.uniform(0, 4, n_train + n_test))
Y_all = 3 + 2*X_all - 0.5*X_all**2 + np.random.normal(0, 0.8, n_train + n_test)

X_tr, X_te = X_all[:n_train].reshape(-1,1), X_all[n_train:].reshape(-1,1)
Y_tr, Y_te = Y_all[:n_train], Y_all[n_train:]

degrees = range(1, 16)
train_mse, test_mse = [], []

for d in degrees:
    pipe = Pipeline([('poly', PolynomialFeatures(d)), ('lr', LinearRegression())])
    pipe.fit(X_tr, Y_tr)
    train_mse.append(mean_squared_error(Y_tr, pipe.predict(X_tr)))
    test_mse.append(mean_squared_error(Y_te, pipe.predict(X_te)))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(degrees, train_mse, 'o-', color='#1e5fa8', lw=2, label='Training MSE', markersize=5)
ax.plot(degrees, test_mse,  'o-', color='#e85d20', lw=2, label='Test MSE',     markersize=5)
ax.axvline(x=2, color='#1a7a45', lw=1.5, ls='--', alpha=0.7, label='Best complexity (degree 2)')
ax.set_xlabel('Model Complexity (polynomial degree)')
ax.set_ylabel('Mean Squared Error')
ax.set_title('Training MSE always falls — Test MSE has a sweet spot')
ax.legend()
ax.set_ylim(0, max(test_mse)*1.1)
plt.tight_layout()
plt.show()

best_degree = degrees[test_mse.index(min(test_mse))]
print(f"\n📌 Best degree by test MSE: {best_degree}")
print(f"   Training MSE at degree 15: {train_mse[-1]:.3f}  ← looks great")
print(f"   Test MSE at degree 15:     {test_mse[-1]:.3f}  ← much worse — overfitting!")

## 🔍 Part 5 — Supervised vs Unsupervised Learning

**Supervised learning** — every observation has both a predictor X *and* a labeled response Y.
- *Regression:* Y is continuous (predict house price, sales, temperature)
- *Classification:* Y is categorical (spam/not spam, disease/healthy, A/B/C)

**Unsupervised learning** — we have X but *no Y*. The goal is to find structure.
- *Clustering:* group similar observations (customer segments, gene expression patterns)
- *Dimensionality reduction:* find a lower-dimensional representation (PCA)

Most of this course is supervised learning (Chapters 2–13 of ISLP). The unsupervised section covers clustering and PCA (Chapter 12).

In [ ]:
# Visual comparison: supervised vs unsupervised
np.random.seed(7)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Supervised — labeled classes
n_each = 60
X_sup = np.vstack([
    np.random.multivariate_normal([1,1], [[0.4,0],[0,0.4]], n_each),
    np.random.multivariate_normal([3,3], [[0.4,0],[0,0.4]], n_each),
    np.random.multivariate_normal([1,3.5], [[0.3,0],[0,0.3]], n_each),
])
y_sup = np.array(['Class A']*n_each + ['Class B']*n_each + ['Class C']*n_each)
colors_sup = {'Class A':'#1e5fa8','Class B':'#e85d20','Class C':'#1a7a45'}
for cls, color in colors_sup.items():
    mask = y_sup == cls
    axes[0].scatter(X_sup[mask,0], X_sup[mask,1], color=color, label=cls, s=30, alpha=0.7)
axes[0].set_title('Supervised Learning\n(labels known — predict the class)')
axes[0].legend()
axes[0].set_xlabel('X₁'); axes[0].set_ylabel('X₂')

# Unsupervised — no labels
axes[1].scatter(X_sup[:,0], X_sup[:,1], color='#888', s=30, alpha=0.5)
axes[1].set_title('Unsupervised Learning\n(no labels — find the structure yourself)')
axes[1].set_xlabel('X₁'); axes[1].set_ylabel('X₂')

plt.suptitle('Same data, different goals', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("📌 In supervised learning we know the answer during training.")
print("   In unsupervised learning we discover structure without any labels.")

## 💼 Exercise — Apply to Real Data

Using the Advertising dataset loaded above, answer the following questions *in code*.

**Task 1:** Which single advertising channel (TV, radio, or newspaper) has the highest correlation with sales? Confirm with a number.

**Task 2:** Fit a simple linear regression (`sklearn.linear_model.LinearRegression`) predicting Sales from TV spend. Split 80/20 train/test. Report training MSE and test MSE.

**Task 3:** Now fit a degree-3 polynomial on the same TV→Sales data. Does test MSE improve or worsen? Why?

**Task 4:** Is predicting sales from advertising spend a *prediction* problem, an *inference* problem, or both? Write 2 sentences explaining your reasoning.

In [ ]:
# ── Exercise workspace ──────────────────────────────────────────────────────

# Task 1: correlation
# YOUR CODE HERE

# Task 2: linear regression + train/test MSE
# YOUR CODE HERE

# Task 3: polynomial degree-3
# YOUR CODE HERE

# Task 4: written answer (as a print statement or comment)
# YOUR CODE HERE

## 🧪 Quiz — 5 Questions

Run the cell below after filling in your answers.

In [ ]:
# @title 📝 Quiz — Statistical Learning
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** In Y = f(X) + ε, what does ε represent?
# @markdown **Q2:** Why does training MSE always decrease as model complexity increases?
# @markdown **Q3:** Name one example of a supervised learning problem
# @markdown **Q4:** Name one example of an unsupervised learning problem
# @markdown **Q5:** Can irreducible error be eliminated by using a more complex model? (yes/no + 1 sentence why)

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results